# Step 2 · Evaluation harness

**Status:** Completed; 47 tests passed in the latest local review.

Verify the same metric, exact ID coverage, word-held-out folds and official scorer agreement. CPU only. This reruns verification rather than a language model.

Start with a fresh runtime and execute cells in order. Read each section before running it. The original combined notebook and its outputs are retained as historical evidence.

## 1. Connect to the project
CPU is sufficient unless you explicitly enable new model inference. This uses the existing Drive project; it does not delete or recopy it.

In [ ]:
from pathlib import Path
import sys, os
try:
    from google.colab import drive
except ImportError:
    PROJECT = Path.cwd().resolve()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/semeval2027/semeval2027_colab')
assert (PROJECT/'src/semeval_task3/data.py').is_file(), f'Edit PROJECT to your project root: {PROJECT}'
assert (PROJECT/'notebooks/project_utils.py').is_file(), 'Use the organized project, including notebooks/project_utils.py.'
sys.path.insert(0, str(PROJECT/'src'))
sys.path.insert(0, str(PROJECT/'notebooks'))
os.environ['PYTHONPATH'] = str(PROJECT/'src')
from project_utils import (run_checked, new_run, save_json, read_scores, save_scores,
                           frozen, standardized_by_word, record_environment, save_evaluation, digest)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, word_bootstrap, read_submission
print('Project:', PROJECT)
print('Existing files stay in place. New outputs use unique directories under outputs/runs/.')

## 2. Install scorer/test dependencies if needed
This cell uses the package registry and records the resulting environment. The original local verification lock is preserved; Colab has a different Python environment.

In [ ]:
run_checked([sys.executable,'-m','pip','install','--quiet','scikit-learn','scipy','bcubed==1.5','pytest'],cwd=PROJECT)
RUN_DIR = new_run(PROJECT, 'step02_evaluation')
record_environment(PROJECT,RUN_DIR/'manifest.json',purpose='Official scorer and test verification')

## 3. Run tests — stop immediately on any failure

In [ ]:
run_checked([sys.executable,'-m','pytest','-q','-p','no:cacheprovider'],cwd=PROJECT)

## 4. Check folds and the official all-positive score

In [ ]:
from semeval_task3.cv import nested_folds
from semeval_task3.evaluation import official_score
from semeval_task3.__main__ import check_agreement
dev = load_dataset(PROJECT/'work/data/dev',with_gold=True)
folds = nested_folds(dev)
assert len(folds)==10
assert all(len(f['train_groups'])==9 and len(f['inner'])==9 for f in folds)
prediction_path = PROJECT/'outputs/step2-verification/all_positive.zip'
local = score(dev.gold,read_submission(prediction_path,dev.keys))
reference = official_score(prediction_path,PROJECT/'work/data/dev',PROJECT/'work/official-scorer')
check_agreement(local,reference)
print('All-positive F1:',local['average'])
print('Official scorer agrees. Ten outer folds, nine inner folds per outer fold.')
save_json(RUN_DIR/'official_agreement.json',dict(local=local,official=reference,folds=folds))
print('Saved:',RUN_DIR)

## Interpretation
 The score is positive-class F1 per word, averaged within language, then equally across languages. It is not accuracy or example-pooled F1. Thresholds use only the nine outer-training words. Paired comparisons resample matching words.

 The organizer clone in Drive may omit `.git`; this notebook directly calls its scoring files and records their hashes, so verification does not depend on Git metadata. Historical scorer commit: `9763564cd540fda979d84ecaa09a6a0e19b9645b`.

 **Next:** Step 3.